In [1]:
!pip install fastapi uvicorn python-multipart python-jose pyngrok ultralytics pandas gdown
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from ultralytics import YOLO
from pyngrok import ngrok
import uvicorn
import threading
import os
import pandas as pd
from PIL import Image
import io
import base64

print("--- ĐANG TỰ ĐỘNG KHỞI TẠO VÀ CẤP PHÁT DỮ LIỆU TỪ KHO LƯU TRỮ ---")
if not os.path.exists('/content/best.pt'):
    !gdown --id "1_r7_KokjECzxu3aoV9Rq4EhMF6SEfE3K" -O /content/best.pt

if not os.path.exists('/content/test_images'):
    !gdown --id "1RApWy6-sPhaZItBUDD9xjQT8iF-CGNfv" -O /content/test_images.zip
    !unzip -q /content/test_images.zip -d /content/test_images/

app = FastAPI()
model = YOLO('/content/best.pt')
DB_DRIVE_FOLDER = '/content/test_images'
CSV_HISTORY_PATH = '/content/pest_test_history.csv'

@app.get("/sync_cloud_dashboard")
async def sync_cloud_dashboard():
    try:
        if os.path.exists(CSV_HISTORY_PATH):
            df = pd.read_csv(CSV_HISTORY_PATH)
            df['MauAnh'] = df['MauAnh'].astype(str)
        else:
            df = pd.DataFrame(columns=["MauAnh", "SoLuong"])

        # Quét sâu đệ quy tìm ảnh ở mọi ngóc ngách của file ZIP
        image_paths = []
        for root, dirs, files in os.walk(DB_DRIVE_FOLDER):
            for file in files:
                if file.lower().endswith(('.jpg', '.png', '.jpeg')):
                    image_paths.append(os.path.join(root, file))

        def get_file_number(filepath):
            filename = os.path.basename(filepath)
            name_without_ext = os.path.splitext(filename)[0]
            if name_without_ext.isdigit():
                return int(name_without_ext)
            return 9999

        image_paths = sorted(image_paths, key=get_file_number)
        has_new_updates = False

        # Tiến trình quét nhận diện và thống kê
        for img_path in image_paths:
            filename = os.path.basename(img_path)
            img_id = os.path.splitext(filename)[0]

            if str(img_id) in df['MauAnh'].values:
                continue

            image = Image.open(img_path)
            results = model.predict(image, imgsz=960, conf=0.45, iou=0.4)
            pest_count = len(results[0].boxes)

            new_record = pd.DataFrame({"MauAnh": [str(img_id)], "SoLuong": [pest_count]})
            df = pd.concat([df, new_record], ignore_index=True)
            has_new_updates = True

        if has_new_updates:
            df['SortKey'] = df['MauAnh'].apply(lambda x: int(x) if x.isdigit() else 9999)
            df = df.sort_values(by="SortKey").drop(columns=["SortKey"])
            df.to_csv(CSV_HISTORY_PATH, index=False)

        # Tiến trình xử lý đồ họa đường biên và mã hóa
        all_annotated_images_b64 = {}
        for img_path in image_paths:
            filename = os.path.basename(img_path)
            img_id = os.path.splitext(filename)[0]

            image = Image.open(img_path)
            results = model.predict(image, imgsz=960, conf=0.45, iou=0.4)

            res_plotted = results[0].plot()
            annotated_image = Image.fromarray(res_plotted[..., ::-1])

            img_byte_arr = io.BytesIO()
            annotated_image.save(img_byte_arr, format='JPEG')
            encoded_img = base64.b64encode(img_byte_arr.getvalue()).decode('utf-8')

            all_annotated_images_b64[str(img_id)] = encoded_img

        return JSONResponse(content={
            "history": df.to_dict(orient="records"),
            "annotated_images": all_annotated_images_b64
        })
    except Exception as e:
        print(f"LỖI HỆ THỐNG: {str(e)}")
        return JSONResponse(status_code=500, content={"message": f"Server Error: {str(e)}"})

# Kích hoạt cổng kết nối Ngrok
# Token ngrok KHÔNG được ghi trực tiếp trong code (tránh lộ khi đẩy lên GitHub).
# Cách 1: Colab -> biểu tượng chìa khóa (Secrets) -> thêm secret tên NGROK_AUTH_TOKEN, bật "Notebook access".
# Cách 2: nếu không có secret, notebook sẽ hỏi bạn dán token khi chạy.
try:
    from google.colab import userdata
    NGROK_TOKEN = userdata.get("NGROK_AUTH_TOKEN")
except Exception:
    from getpass import getpass
    NGROK_TOKEN = getpass("Nhập ngrok authtoken: ")
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(8000).public_url
print(f"COPY ĐƯỜNG LINK API CLOUD MỚI NÀY VÀO APP.PY: {public_url}/sync_cloud_dashboard")

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run_server, daemon=True).start()